# 02 — Evaluation
Scores the OCR pipeline's output: structural checks, propagation flags, ground-truth comparison.


## Imports & Drive helpers


In [ ]:
from config import (
    OCR_RESULTS_DIR,
    LOGS_DIR,
    EVAL_OUTPUT_DIR,
    GROUND_TRUTH_DIR,
    DRIVE_ROOT_FOLDER,
    DRIVE_OCR_FOLDER,
    DRIVE_EVALUATION_FOLDER,
    SAVE_MODE,
)

import csv
import io
import tempfile
import re
from pathlib import Path
import pandas as pd
from collections import Counter, defaultdict
from Levenshtein import distance as levenshtein_distance
from googleapiclient.http import MediaFileUpload
from datetime import datetime


# DRIVE AUTH + HELPERS

from drive_utils import (
    get_drive_service,
    get_or_create_folder as get_or_create_drive_folder,
    find_folder,
    list_files_in_folder,
    download_text,
    drive_file_exists,
)

## Configuration


In [ ]:
ROOT_FOLDER_NAME = DRIVE_ROOT_FOLDER       # "Venezia_Almanac_1947"
RESULTS_FOLDER   = DRIVE_OCR_FOLDER        # "llm_ocr_results"
EVAL_FOLDER_NAME = DRIVE_EVALUATION_FOLDER # "evaluation"

PAGES = list(range(494, 571)) + [41, 130, 161, 170, 320, 367, 383, 458]   # "all" | [6, 7, 8] | range(100, 200)
# Controls which pages are evaluated in this run.
# The output folder name encodes the scope (eval_all_... vs eval_pages_...) so the version selector knows whether to use this eval for full selection.

USE_FIXED_SCHEMA = False
# If True, maps all CSVs to a canonical schema before comparison (ignores column naming differences, focuses on semantic content).
# If False, uses original extracted schemas for stricter evaluation.

# Local paths 
LOCAL_ROOT = Path(OCR_RESULTS_DIR)   
LOCAL_LOGS = Path(LOGS_DIR)        
LOCAL_EVAL = Path(EVAL_OUTPUT_DIR)  


In [ ]:

# LOG LOADING + PARSING

def load_run_logs(service, root_id, results_id=None):
    # loads all run_N_scope.log files, local first then Drive per SAVE_MODE
    if SAVE_MODE in ("local", "both"):
        if LOCAL_LOGS.exists():
            log_files_local = list(LOCAL_LOGS.glob("run_*.log"))
            if log_files_local:
                print(f"Loading logs from local "
                      f"({len(log_files_local)} files): {LOCAL_LOGS}")
                log_data = {}
                for f in log_files_local:
                    name    = f.name
                    content = f.read_text(encoding="utf-8")
                    m = re.match(r'^run_(\d+)_(.+)\.log$', name)
                    if m:
                        log_data[(int(m.group(1)), m.group(2))] = (name, content)
                    else:
                        log_data[(0, name)] = (name, content)
                return log_data
            elif SAVE_MODE == "local":
                print(f"WARNING: no log files found in {LOCAL_LOGS}")
                return {}
            # SAVE_MODE == "both" and no local logs — fall through to Drive

    if SAVE_MODE in ("drive", "both"):
        logs_subfolder = find_folder(service, "logs", root_id)
        if logs_subfolder:
            folder_id    = logs_subfolder
            folder_label = "logs/ subfolder under root"
        else:
            folder_id    = root_id
            folder_label = "root folder (Venezia_Almanac_1947)"

        print(f"Loading logs from Drive: {folder_label}")

        try:
            all_files = list_files_in_folder(service, folder_id)
        except Exception as e:
            print(f"ERROR listing logs folder: {e}")
            return {}

        log_files = [f for f in all_files if f["name"].endswith(".log")]
        if not log_files:
            print(f"WARNING: no .log files found in {folder_label}")
            return {}

        log_data = {}
        for f in log_files:
            name    = f["name"]
            content = download_text(service, f["id"])
            m = re.match(r'^run_(\d+)_(.+)\.log$', name)
            if m:
                log_data[(int(m.group(1)), m.group(2))] = (name, content)
                continue
            legacy = re.match(r'^(?:run_|log_)?(\d+)(?:\.log)?$', name)
            if legacy:
                log_data[(int(legacy.group(1)), "unknown")] = (name, content)
                continue
            log_data[(0, name)] = (name, content)
        return log_data

    return {}


def parse_run_log(log_text, run_id):
    # run log text -> {page_num: {layout, ad_type, warnings, schema, flags}}
    pages = {}
    current_page = None

    for line in log_text.splitlines():
        if "Processing: page_" in line:
            m = re.search(r'page_(\d+)', line)
            if m:
                current_page = int(m.group(1))
                pages[current_page] = {
                    "run_id":   run_id,
                    "warnings": [],
                    "layout":   None,
                    "ad_type":  None,
                    "schema":   None,
                    "flags":    []
                }
        if current_page is None:
            continue
        if "Layout type:" in line:
            pages[current_page]["layout"] = line.split(":")[-1].strip()
        if "Ad type:" in line:
            pages[current_page]["ad_type"] = line.split(":")[-1].strip()
        if "Inferred schema:" in line:
            pages[current_page]["schema"] = line.split(":")[-1].strip()
        if "WARNING:" in line:
            pages[current_page]["warnings"].append(line.strip())
        if "Section state carries address" in line:
            pages[current_page]["flags"].append("propagation_address")
        if "OCR refusal detected" in line:
            pages[current_page]["flags"].append("ocr_refusal")

    return pages


def _pages_from_scope(scope: str):
    # scope string -> expected set of page numbers, or None if unknown
    if scope in ("all", "unknown"):
        return None
    range_m = re.match(r'^range_(\d+)_(\d+)$', scope)
    if range_m:
        return set(range(int(range_m.group(1)), int(range_m.group(2)) + 1))
    pages_m = re.match(r'^pages_([\d_]+)$', scope)
    if pages_m:
        return set(int(n) for n in pages_m.group(1).split("_") if n)
    return None


def build_log_index(log_data):
    # {(run_num, scope): {page_num: log_meta}} index + page_run_map
    log_index    = {}
    log_issues   = []
    page_run_map = defaultdict(list)

    for (run_num, scope), (name, text) in log_data.items():
        parsed = parse_run_log(text, run_num)

        page_count = len(parsed)
        if page_count == 0:
            log_issues.append(f"LOG ANOMALY: {name} parsed 0 pages — check format")
        elif page_count < 5:
            log_issues.append(
                f"LOG ANOMALY: {name} only has {page_count} pages — possibly incomplete"
            )

        lines = text.splitlines()
        dates_found = [l for l in lines if re.search(r'\d{4}-\d{2}-\d{2}', l)]
        if dates_found:
            first_date = re.search(r'\d{4}-\d{2}-\d{2}', dates_found[0]).group()
            last_date  = re.search(r'\d{4}-\d{2}-\d{2}', dates_found[-1]).group()
            if first_date != last_date:
                log_issues.append(
                    f"LOG NOTE: {name} spans {first_date} to {last_date}"
                )

        actual_pages        = set(parsed.keys())
        expected_from_scope = _pages_from_scope(scope)
        if expected_from_scope is not None:
            unexpected = actual_pages - expected_from_scope
            missing    = expected_from_scope - actual_pages
            if unexpected:
                log_issues.append(
                    f"LOG NOTE: {name} contains pages outside its scope: "
                    f"{sorted(unexpected)[:5]}"
                    f"{'...' if len(unexpected) > 5 else ''}"
                )
            if missing:
                log_issues.append(
                    f"LOG NOTE: {name} missing {len(missing)} pages from its scope "
                    f"(first few: {sorted(missing)[:5]})"
                )

        log_index[(run_num, scope)] = parsed
        for page_num in parsed:
            page_run_map[page_num].append(run_num)

    if not any(
        any(494 <= p <= 570 for p in pdict.keys())
        for pdict in log_index.values()
    ):
        log_issues.append("LOG NOTE: no log covers provincia pages (494-570)")

    return log_index, log_issues, page_run_map


def get_best_log_meta(log_index, run_id, page_num):
    # log metadata for a run_id/page_num, exact match first then scope/highest-run fallback
    for (log_run, scope), page_dict in log_index.items():
        if log_run == run_id and page_num in page_dict:
            return page_dict[page_num]

    for (log_run, scope), page_dict in log_index.items():
        expected = _pages_from_scope(scope)
        if expected is not None and page_num in expected and page_num in page_dict:
            return page_dict[page_num]

    candidates = [
        (log_run, page_dict[page_num])
        for (log_run, scope), page_dict in log_index.items()
        if isinstance(log_run, int) and page_num in page_dict
    ]
    if candidates:
        return max(candidates, key=lambda x: x[0])[1]

    return {}

## Scoring building blocks
Normalization, structural/propagation checks, ground-truth comparison, provincia eval, failure classification.


In [ ]:

# NORMALIZATION + METRICS

def normalize(text):
    if text is None:
        return ""
    text = str(text)
    for ch in ['\u201c', '\u201d', '\u00ab', '\u00bb', '"', "'"]:
        text = text.replace(ch, '')
    text = text.replace('[', '').replace(']', '')
    return " ".join(text.split()).lower()


# Levenshtein-based character similarity.
def char_accuracy(t1, t2):
    dist = levenshtein_distance(t1, t2)
    m = max(len(t1), len(t2))
    return 100.0 if m == 0 else (1 - dist / m) * 100


def word_accuracy(t1, t2):
    # not used in main metrics -- positional token comparison is unreliable for unordered lists, kept for reference
    w1, w2 = t1.split(), t2.split()
    matches = sum(1 for a, b in zip(w1, w2) if a == b)
    total = max(len(w1), len(w2))
    return 100.0 * matches / total if total else 100


# Bag-of-words overlap between two texts.
def bow_accuracy(t1, t2):
    c1 = Counter(normalize(t1).split())
    c2 = Counter(normalize(t2).split())
    common = sum((c1 & c2).values())
    total = sum((c1 | c2).values())
    return 100.0 if total == 0 else common / total * 100


In [ ]:
# Level 1 (all pages): CSV exists, right columns, no empty/truncated rows.
# Extra columns like 'page' or 'Additional Info' are expected, not errors.

# SCHEMA + STRUCTURAL CHECKS

TITLE_ONLY = re.compile(
    r'^(dott|rag|ing|prof|avv|cav|dr|sig|comm|geom)\.?$', re.I
)

EXPECTED_SCHEMAS = {
    range(14, 36):   {"Category", "Page"},
    range(36, 38):   {"Name", "Address", "Location", "Role", "Additional Info"},
    range(42, 56):   {"Name", "Address", "Location", "Role", "Profession"},
    range(86, 98):   {"Name", "Address", "Location", "Role", "Additional Info"},
    range(108, 129): {"Name", "Address", "Location", "Profession", "Additional Info"},
    range(132, 319): {"Name", "Address", "Location", "Category", "Additional Info"},
    range(332, 488): {"Name", "Address", "Location", "Category", "Additional Info"},
    range(494, 571): {"Provincia", "Section", "Role_or_Profession", "Name", "Address"},
}

ALWAYS_ALLOWED_COLS = {"page", "Additional Info", "Altre_info"}

SKIP_PAGES     = {1, 2, 6, 7, 8, 9, 490, 571}
INDEX_PAGES    = set(range(10, 36)) | {491, 492, 493}
PROVINCIA_PAGES = set(range(494, 571))
AD_ONLY_RANGES = [
    (3, 5), (11, 13), (20, 21), (23, 23), (27, 27), (30, 31),
    (40, 41), (46, 47), (105, 107), (129, 131), (319, 331), (572, 575)
]

# Checks the page number against hardcoded full-ad ranges.
def is_full_ad_page(page_num):
    if not page_num:
        return False
    for start, end in AD_ONLY_RANGES:
        if start <= page_num <= end:
            return True
    return False

PROVINCIA_INFO_SCHEMA = {
    "Provincia", "Frazioni", "Abitanti", "Superficie",
    "Stazione", "Prodotti", "Altre_info", "page"
}
PROVINCIA_ADS_SCHEMA = {
    "Name", "Address", "Category", "Additional Info", "page"
}


# Expected CSV columns for a given page range.
def expected_schema(page):
    for r, s in EXPECTED_SCHEMAS.items():
        if page in r:
            return s
    return None


def get_page_content_type(page_num):
    # content category for a page number: skip / index / provincia / structured
    if page_num is None:
        return "structured"
    if page_num in SKIP_PAGES:
        return "skip"
    if page_num in INDEX_PAGES:
        return "index"
    if page_num in PROVINCIA_PAGES:
        return "provincia"
    return "structured"


# Structural checks on one semantic CSV: missing columns, empty rows, truncated names.
def analyze_csv(text, page_num):
    issues = []
    rows = list(csv.reader([l for l in text.splitlines() if l.strip()]))
    if len(rows) < 2:
        return {"issues": ["EMPTY_FILE"]}

    header    = rows[0]
    hset      = {h.strip() for h in header}
    hset_lower = {h.lower() for h in hset}
    content_type = get_page_content_type(page_num)

    if content_type not in ("skip", "index"):
        expected = expected_schema(page_num)
        if expected:
            expected_lower = {e.lower() for e in expected}
            missing = {e for e in expected if e.lower() not in hset_lower}
            extra = {
                h for h in hset
                if h.lower() not in expected_lower
                and h.lower() not in {a.lower() for a in ALWAYS_ALLOWED_COLS}
            }
            if missing:
                issues.append(f"missing_cols:{missing}")
            if extra:
                issues.append(f"extra_cols:{extra}")

    empty_rows = sum(1 for r in rows[1:] if not any(x.strip() for x in r))
    trunc = sum(1 for r in rows[1:] if r and TITLE_ONLY.match(r[0].strip()))

    return {
        "issues":           issues,
        "rows":             len(rows) - 1,
        "empty_rows":       empty_rows,
        "truncated_names":  trunc,
        "columns":          hset,
    }


In [ ]:
# Level 2 (all pages, manual follow-up): flags pages where propagation may have failed. 

# PROPAGATION SIGNAL CHECKS

def get_val(row, target):
    for k, v in row.items():
        if k and k.strip().lower() == target.lower():
            if isinstance(v, list):
                return " ".join(str(x) for x in v if x)
            return str(v).strip() if v else ""
    return ""

def check_propagation_signal(csv_text, page_num):
    # flags pages where >10% of rows have neither Address nor Location
    result = {
        "propagation_flag": False,
        "empty_address_rows": 0,
        "empty_address_pct": 0.0,
        "total_data_rows": 0,
        "addr_only_missing": 0,
        "loc_only_missing": 0,
        "note": ""
    }

    content_type = get_page_content_type(page_num)
    if content_type in ("skip", "index"):
        result["note"] = f"skipped — page type: {content_type}"
        return result
    if content_type == "provincia":
        result["note"] = "skipped — use check_provincia_signal"
        return result
    # Full ad pages have no Location column by design.
    # Missing addresses here are a real extraction failure (not propagation).
    if is_full_ad_page(page_num):
        result["note"] = "skipped — full ad page, address failure tracked separately"
        return result

    lines = [l for l in csv_text.split("\n") if l.strip()]
    if len(lines) < 2:
        return result

    try:
        rows = list(csv.DictReader(lines))
    except Exception:
        return result
    if not rows:
        return result

    has_address  = any(k and k.strip().lower() == "address"  for k in rows[0].keys())
    has_location = any(k and k.strip().lower() == "location" for k in rows[0].keys())
    if not has_address and not has_location:
        return result

    both_missing = addr_only_missing = loc_only_missing = total_data_rows = 0

    for row in rows:
        name_val = get_val(row, "Name").strip()
        if not name_val or name_val == "__SECTION__":
            continue
        total_data_rows += 1
        addr_val = get_val(row, "Address") if has_address else ""
        loc_val  = get_val(row, "Location") if has_location else ""
        if not addr_val and not loc_val:
            both_missing += 1
        elif not addr_val:
            addr_only_missing += 1
        elif not loc_val:
            loc_only_missing += 1

    result["total_data_rows"]   = total_data_rows
    result["addr_only_missing"] = addr_only_missing
    result["loc_only_missing"]  = loc_only_missing

    if total_data_rows == 0:
        return result

    both_pct = 100 * both_missing / total_data_rows
    if both_pct > 10:
        result["propagation_flag"]   = True
        result["empty_address_rows"] = both_missing
        result["empty_address_pct"]  = round(both_pct, 1)
        result["note"] = (
            f"{both_missing}/{total_data_rows} rows ({both_pct:.0f}%) "
            f"have neither Address nor Location — "
            f"check if propagation filled these"
        )
    else:
        result["note"] = (
            f"{total_data_rows} data rows: "
            f"both missing={both_missing} | "
            f"addr only missing={addr_only_missing} | "
            f"loc only missing={loc_only_missing}"
        )

    return result


def check_provincia_signal(csv_text):
    # flags empty values in the Provincia column of a provincia aggregate file
    result = {
        "propagation_flag": False,
        "empty_address_rows": 0,
        "empty_address_pct": 0.0,
        "note": ""
    }

    lines = [l for l in csv_text.split("\n") if l.strip()]
    if len(lines) < 2:
        return result

    try:
        rows = list(csv.DictReader(lines))
    except Exception:
        return result
    if not rows:
        return result

    has_provincia = any(
        k and k.strip().lower() == "provincia"
        for k in rows[0].keys()
    )
    if not has_provincia:
        result["propagation_flag"] = True
        result["note"] = "Provincia column missing entirely from this file"
        return result

    total_rows = empty_prov = 0
    for row in rows:
        total_rows += 1
        if not get_val(row, "Provincia"):
            empty_prov += 1

    if total_rows == 0:
        return result

    empty_pct = 100 * empty_prov / total_rows
    if empty_prov > 0:
        result["propagation_flag"]   = True
        result["empty_address_rows"] = empty_prov
        result["empty_address_pct"]  = round(empty_pct, 1)
        result["note"] = (
            f"{empty_prov}/{total_rows} rows ({empty_pct:.0f}%) "
            f"have empty Provincia field — every row should have a town name"
        )
    return result


In [ ]:
# Level 3 (~10 pages with ground truth): char accuracy, bag-of-words, semantic precision/recall/F1, row count similarity.

def load_ground_truth(page_name):
    # loads ground truth .txt + _semantic.csv for a page, always local
    gt_text     = None
    gt_csv_text = None

    gt_dir   = Path(GROUND_TRUTH_DIR)
    txt_path = gt_dir / "txt" / f"{page_name}_ground_truth.txt"
    csv_path = gt_dir / "csv" / f"{page_name}_ground_truth_semantic.csv"

    if txt_path.exists():
        gt_text = txt_path.read_text(encoding="utf-8")
        print(f"  Ground truth txt found: {txt_path.name}")
    if csv_path.exists():
        gt_csv_text = csv_path.read_text(encoding="utf-8")
        print(f"  Ground truth csv found: {csv_path.name}")

    return gt_text, gt_csv_text


def load_ocr_text(service, page_folder_id, page_name, run_file_name):
    # raw OCR .txt matching a semantic CSV, Drive-first then local, falls back to scanning
    run_num_match = re.search(r'_(\d+)\.csv$', run_file_name)
    run_num = run_num_match.group(1) if run_num_match else "1"

    base = re.sub(r'_semantic_\d+\.csv$', '', run_file_name)
    primary_candidate = f"{base}_ocr_{run_num}.txt"

    if SAVE_MODE in ("drive", "both"):
        file_id = drive_file_exists(service, page_folder_id, primary_candidate)
        if file_id:
            print(f"  OCR txt found: {primary_candidate}")
            return download_text(service, file_id)

        all_files = list_files_in_folder(service, page_folder_id)

        txt_files = sorted(
            [f for f in all_files
             if "_ocr_" in f["name"] and f["name"].endswith(".txt")],
            key=lambda x: x.get("createdTime", "")
        )
        for f in txt_files:
            if f["name"].endswith(f"_ocr_{run_num}.txt"):
                print(f"  OCR txt matched: {f['name']}")
                return download_text(service, f["id"])
        if len(txt_files) == 1:
            print(f"  OCR txt fallback (only one found): {txt_files[0]['name']}")
            return download_text(service, txt_files[0]["id"])

        if SAVE_MODE == "drive":
            print(f"  WARNING: no OCR text found for {run_file_name}")
            return None
        # SAVE_MODE == "both" — fall through to local

    if SAVE_MODE in ("local", "both"):
        page_path = LOCAL_ROOT / page_name
        p = page_path / primary_candidate
        if p.exists():
            return p.read_text(encoding="utf-8")

    print(f"  WARNING: no OCR text found for {run_file_name}")
    return None


In [ ]:
# EVALUATION AGAINST GROUND TRUTH

def evaluate_against_ground_truth(ocr_text, gt_text, ocr_csv_text, gt_csv_text):
    # compares OCR output to ground truth: text + CSV metrics
    metrics = {}

    if ocr_text and gt_text:
        n_ocr = normalize(ocr_text)
        n_gt  = normalize(gt_text)
        metrics["char_accuracy"]  = round(char_accuracy(n_ocr, n_gt), 2)
        metrics["bow_similarity"] = round(bow_accuracy(n_ocr, n_gt),  2)
    else:
        metrics["char_accuracy"]  = None
        metrics["bow_similarity"] = None

    if ocr_csv_text and gt_csv_text:
        try:
            ocr_rows = list(csv.DictReader(io.StringIO(ocr_csv_text)))
            gt_rows  = list(csv.DictReader(io.StringIO(gt_csv_text)))
        except Exception as e:
            print(f"  WARNING: could not parse ground truth CSV: {e}")
            ocr_rows, gt_rows = [], []

        if ocr_rows and gt_rows:
            metrics["row_count_similarity"] = round(
                100 * min(len(ocr_rows), len(gt_rows))
                / max(len(ocr_rows), len(gt_rows)), 2
            )
        else:
            metrics["row_count_similarity"] = None

        def safe_cell_values(rows):
            for r in rows:
                for v in r.values():
                    if isinstance(v, list):
                        v = " ".join(str(x) for x in v if x)
                    if v and isinstance(v, str) and v.strip() and v.strip().lower() != "page":
                        yield normalize(v)

        def col_aware_values(rows):
            for r in rows:
                for k, v in r.items():
                    if isinstance(v, list):
                        v = " ".join(str(x) for x in v if x)
                    if v and isinstance(v, str) and v.strip() and v.strip().lower() != "page":
                        yield (k or "").strip().lower(), normalize(v)

        def name_values(rows):
            for r in rows:
                for k, v in r.items():
                    if k and k.strip().lower() == "name":
                        if isinstance(v, list):
                            v = " ".join(str(x) for x in v if x)
                        if v and str(v).strip():
                            yield normalize(v)

        ocr_values = set(safe_cell_values(ocr_rows))
        gt_values  = set(safe_cell_values(gt_rows))
        tp         = len(ocr_values & gt_values)
        precision  = tp / len(ocr_values) if ocr_values else 0
        recall     = tp / len(gt_values)  if gt_values  else 0
        f1 = (2 * precision * recall / (precision + recall)
              if (precision + recall) else 0)
        metrics["semantic_precision"] = round(precision * 100, 2)
        metrics["semantic_recall"]    = round(recall    * 100, 2)
        metrics["semantic_f1"]        = round(f1        * 100, 2)

        ocr_pairs = set(col_aware_values(ocr_rows))
        gt_pairs  = set(col_aware_values(gt_rows))
        tp_col    = len(ocr_pairs & gt_pairs)
        prec_col  = tp_col / len(ocr_pairs) if ocr_pairs else 0
        rec_col   = tp_col / len(gt_pairs)  if gt_pairs  else 0
        f1_col    = (2 * prec_col * rec_col / (prec_col + rec_col)
                     if (prec_col + rec_col) else 0)
        metrics["semantic_f1_col_aware"] = round(f1_col * 100, 2)

        ocr_names = set(name_values(ocr_rows))
        gt_names  = set(name_values(gt_rows))
        metrics["name_recall"] = (
            round(100 * len(ocr_names & gt_names) / len(gt_names), 2)
            if gt_names else None
        )
    else:
        for k in ["row_count_similarity", "semantic_precision", "semantic_recall",
                  "semantic_f1", "semantic_f1_col_aware", "name_recall"]:
            metrics[k] = None

    return metrics




In [ ]:
# Provincia pages (494-570): evaluates the two aggregate files -- provincia_di_venezia_N.csv (town info) and provincia_ads_N.csv (ads).

def evaluate_provincia_aggregates(service, root_id, results, markdown_lines, pages_requested):
    # evaluates provincia_di_venezia_N.csv and provincia_ads_N.csv
    if pages_requested != "all":
        if isinstance(pages_requested, list):
            if not any(494 <= p <= 570 for p in pages_requested):
                return
        elif isinstance(pages_requested, range):
            if not any(494 <= p <= 570 for p in pages_requested):
                return

    markdown_lines.append("\n## Provincia Aggregate Files\n")

    files = []

    if SAVE_MODE in ("drive", "both"):
        results_id          = find_folder(service, RESULTS_FOLDER, root_id)
        provincia_folder_id = find_folder(service, "provincia", results_id)
        search_folder_id    = provincia_folder_id if provincia_folder_id else results_id
        if provincia_folder_id:
            print("  Found provincia/ subfolder")
        else:
            print("  WARNING: no provincia/ subfolder, searching llm_ocr_results root")

        all_drive_files = list_files_in_folder(service, search_folder_id)
        files = [f for f in all_drive_files
                 if "provincia" in f["name"].lower() and f["name"].endswith(".csv")]

    if not files and SAVE_MODE in ("local", "both"):
        local_prov = LOCAL_ROOT / "provincia"
        if local_prov.exists():
            files = [
                {"name": f.name, "path": str(f)}
                for f in local_prov.iterdir()
                if f.is_file() and "provincia" in f.name.lower()
                and f.name.endswith(".csv")
            ]

    if not files:
        markdown_lines.append("- No provincia aggregate files found.\n")
        return

    for f in sorted(files, key=lambda x: x["name"]):
        fname = f["name"]
        if SAVE_MODE in ("drive", "both") and f.get("id"):
            csv_text = download_text(service, f["id"])
        else:
            csv_text = Path(f["path"]).read_text(encoding="utf-8")

        lines  = [l for l in csv_text.split("\n") if l.strip()]
        n_rows = len(lines) - 1 if len(lines) > 1 else 0

        try:
            header_row = next(csv.reader([lines[0]])) if lines else []
            hset = {h.strip() for h in header_row}
        except Exception:
            hset = set()

        is_ads  = "ads" in fname.lower()
        is_info = "venezia" in fname.lower() and not is_ads
        expected_schema_set = PROVINCIA_ADS_SCHEMA if is_ads else PROVINCIA_INFO_SCHEMA

        issues = []
        if n_rows == 0:
            issues.append("EMPTY_FILE")
        missing = expected_schema_set - hset - ALWAYS_ALLOWED_COLS
        extra   = hset - expected_schema_set - ALWAYS_ALLOWED_COLS
        if missing:
            issues.append(f"missing_cols:{missing}")
        if extra:
            issues.append(f"extra_cols:{extra}")

        key_col   = "Provincia" if is_info else "Name"
        empty_key = trunc = 0
        try:
            rows_parsed = list(csv.DictReader(io.StringIO(csv_text)))
            empty_key   = sum(1 for r in rows_parsed if not r.get(key_col, "").strip())
            trunc       = sum(1 for r in rows_parsed
                              if is_ads and TITLE_ONLY.match(r.get("Name", "").strip()))
        except Exception:
            pass

        prop = check_provincia_signal(csv_text) if is_info else check_propagation_signal(csv_text, None)

        results.append({
            "Page":             fname,
            "Run":              "PROVINCIA_AGGREGATE",
            "Rows":             n_rows,
            "Empty Rows":       empty_key,
            "Truncated Names":  trunc,
            "Issues":           "; ".join(issues) if issues else "OK",
            "Propagation Flag": prop["propagation_flag"],
            "Propagation Note": prop["note"],
        })

        status = "OK" if not issues else "; ".join(issues)
        markdown_lines.append(f"- **{fname}**: {n_rows} rows | {status}")
        if prop["propagation_flag"]:
            markdown_lines.append(f"  - ⚠ {prop['note']}")
        markdown_lines.append("")


In [ ]:
# FAILURE CLASSIFICATION

def classify_page_failure(metrics_list, log_meta_list, run_ids=None):
    reasons = []

    total_empty = sum(m.get("empty_rows", 0) for m in metrics_list)
    total_trunc = sum(m.get("truncated_names", 0) for m in metrics_list)
    total_rows  = sum(m.get("rows", 0) for m in metrics_list)

    layouts = {lm.get("layout") for lm in log_meta_list if lm.get("layout")}
    ads     = {lm.get("ad_type") for lm in log_meta_list if lm.get("ad_type")}

    flags = set()
    for lm in log_meta_list:
        flags.update(lm.get("flags", []))

    all_issues = []
    for m in metrics_list:
        all_issues.extend(m.get("issues", []))
    has_missing_cols = any("missing_cols" in i for i in all_issues)

    if total_empty > 2:
        reasons.append("STRUCTURAL_MISSING_ROWS")
    if total_trunc > 0:
        reasons.append("TEXT_TRUNCATION")
    if has_missing_cols:
        reasons.append("SCHEMA_MISMATCH")
    if "COMPLEX_LAYOUT" in layouts and (
        total_empty > 2 or total_trunc > 0 or has_missing_cols
    ):
        reasons.append("COMPLEX_LAYOUT_WITH_ERRORS")
    if "FULL_AD" in ads and total_rows < 3:
        reasons.append("AD_PAGE_LOW_OUTPUT")
    elif "PARTIAL_AD" in ads and (total_empty > 2 or total_trunc > 0 or has_missing_cols):
        reasons.append("AD_INTERFERENCE_WITH_ERRORS")
    if "ocr_refusal" in flags:
        if run_ids:
            failed_runs = [
                str(run_ids[i])
                for i, lm in enumerate(log_meta_list)
                if "ocr_refusal" in lm.get("flags", [])
                and i < len(run_ids)
            ]
            reasons.append(
                f"OCR_FAILURE(runs:{','.join(failed_runs)})"
                if failed_runs else "OCR_FAILURE"
            )
        else:
            reasons.append("OCR_FAILURE")
    if "propagation_address" in flags and total_empty > 0:
        reasons.append("PROPAGATION_ERROR")
    if "FULL_AD" in ads:
        # Check if addresses were poorly extracted from ad content.
        # This is a real OCR failure, distinct from propagation failure.
        avg_missing = (
            sum(
                m.get("propagation_signal", {}).get("empty_address_pct", 0)
                for m in metrics_list
            ) / max(len(metrics_list), 1)
        )
        if avg_missing > 50:
            reasons.append("ADDRESS_EXTRACTION_LOW")
        elif not reasons:
            reasons.append("FULL_AD_PAGE")

    return reasons


# SUMMARY TABLE + REPORT BUILDERS

def _page_sort_key(page_name):
    m = re.search(r'(\d+)', page_name)
    return int(m.group(1)) if m else 999999


# Recommends a triage action (rerun, check schema, etc.) from a page's failure flags.
def _suggest_action(d, row_min, row_max, n_runs):
    if d["missing"]:
        return "RERUN"
    if not d["schema_ok"]:
        return "CHECK SCHEMA"
    if d["prop_flag"]:
        return "CHECK PROPAGATION"
    if row_max - row_min > 10 and n_runs > 1:
        return "CHECK ROW COUNT"
    if d["has_gt"] and d["f1_scores"] and max(d["f1_scores"]) < 60:
        return "VERIFY GT"
    if d["failure_reasons"] and "FULL_AD_PAGE" not in d["failure_reasons"]:
        return "CHECK FAILURE"
    return "OK"


def build_summary_table(results, markdown_lines):
    # one-row-per-page summary, sorted by action priority
    page_data = defaultdict(lambda: {
        "runs": [],
        "rows": [],
        "schema_ok": True,
        "prop_flag": False,
        "prop_note": "",
        "has_gt": False,
        "f1_scores": [],
        "char_acc_scores": [],
        "failure_reasons": "",
        "missing": False,
        "partial_missing": 0,
        "col_diffs": [],
    })

    for r in results:
        page = r.get("Page", "")
        run  = r.get("Run", "")
        if not page:
            continue
        if run == "MISSING":
            # Only mark as truly missing if there are NO semantic CSVs at all.
            page_data[page]["partial_missing"] = page_data[page].get("partial_missing", 0) + 1
            continue
        if run == "PAGE_SUMMARY":
            page_data[page]["failure_reasons"] = r.get("Failure Reasons", "")
            continue
        if run.startswith("compare_"):
            col_diff = r.get("Column Differences", "")
            if col_diff and col_diff not in ("none", ""):
                page_data[page]["col_diffs"].append(col_diff)
            continue
        if run == "PROVINCIA_AGGREGATE":
            continue

        page_data[page]["runs"].append(run)
        page_data[page]["rows"].append(r.get("Rows", 0))
        if "missing_cols" in str(r.get("Issues", "")):
            page_data[page]["schema_ok"] = False
        if r.get("Propagation Flag"):
            page_data[page]["prop_flag"] = True
            page_data[page]["prop_note"] = r.get("Propagation Note", "")
        if r.get("semantic_f1") is not None:
            page_data[page]["has_gt"] = True
            page_data[page]["f1_scores"].append(r["semantic_f1"])
        if r.get("char_accuracy") is not None:
            page_data[page]["char_acc_scores"].append(r["char_accuracy"])

    summary_rows = []
    for page, d in sorted(page_data.items(), key=lambda x: _page_sort_key(x[0])):
        n_runs  = len(d["runs"])
        row_min = min(d["rows"]) if d["rows"] else 0
        row_max = max(d["rows"]) if d["rows"] else 0
        # A page is truly missing only if it has zero semantic CSVs.
        # If it has some runs but some OCR-without-CSV gaps, that is partial_missing (info only)
        if d["partial_missing"] > 0 and n_runs == 0:
            d["missing"] = True
        action  = _suggest_action(d, row_min, row_max, n_runs)
        summary_rows.append({
            "Page":            page,
            "Runs Found":      n_runs,
            "Rows Min":        row_min,
            "Rows Max":        row_max,
            "Row Variation":   row_max - row_min,
            "Schema OK":       "✓" if d["schema_ok"] else "✗",
            "Prop Flag":       "YES" if d["prop_flag"] else "",
            "Prop Note":       d["prop_note"],
            "Schema Drift":    "; ".join(d["col_diffs"]) if d["col_diffs"] else "",
            "Has GT":          "✓" if d["has_gt"] else "",
            "Best F1":         max(d["f1_scores"]) if d["f1_scores"] else None,
            "Best Char Acc":   max(d["char_acc_scores"]) if d["char_acc_scores"] else None,
            "Failure Reasons": d["failure_reasons"],
            "Missing":         "✗" if d["missing"] else "",
            "OCR Only Runs":   d["partial_missing"] if d["partial_missing"] > 0 else "",
            "Action":          action,
        })

    if not summary_rows:
        return pd.DataFrame()

    df = pd.DataFrame(summary_rows)
    priority = {
        "RERUN": 0, "CHECK SCHEMA": 1, "CHECK PROPAGATION": 2,
        "CHECK ROW COUNT": 3, "VERIFY GT": 4, "CHECK FAILURE": 5, "OK": 6,
    }
    df["_sort"] = df["Action"].map(lambda a: priority.get(a, 9))
    df = df.sort_values(["_sort", "Page"]).drop(columns=["_sort"]).reset_index(drop=True)
    return df


def save_reports(results, md_text, run_name, summary_df=None):
    # saves report files locally (temp dir if SAVE_MODE=drive), returns paths for upload
    df = pd.DataFrame(results)

    base_dir = (
        LOCAL_EVAL if SAVE_MODE in ("local", "both")
        else Path(tempfile.mkdtemp())
    )

    file_paths = {
        "csv":     base_dir / f"{run_name}_results.csv",
        "md":      base_dir / f"{run_name}_report.md",
        "summary": base_dir / f"{run_name}_page_summary.csv",
    }

    df.to_csv(file_paths["csv"], index=False)
    file_paths["md"].write_text(md_text, encoding="utf-8")
    if summary_df is not None:
        summary_df.to_csv(file_paths["summary"], index=False)

    return {k: str(v) for k, v in file_paths.items()}


def save_evaluation_reports(service, files, run_name):
    # saves evaluation reports locally and/or to Drive, same folder name on both
    if SAVE_MODE in ("local", "both"):
        local_eval_dir = LOCAL_EVAL / run_name
        local_eval_dir.mkdir(parents=True, exist_ok=True)
        for key, fpath in files.items():
            src = Path(fpath)
            dst = local_eval_dir / src.name
            if src != dst:
                dst.write_bytes(src.read_bytes())
        print(f"Reports saved locally: {local_eval_dir}/")

    if SAVE_MODE in ("drive", "both") and service:
        root_id = find_folder(service, ROOT_FOLDER_NAME)
        eval_id = find_folder(service, EVAL_FOLDER_NAME, root_id)
        if not eval_id:
            eval_id = get_or_create_drive_folder(
                service, EVAL_FOLDER_NAME, root_id
            )
        run_folder_id = service.files().create(
            body={
                "name": run_name,
                "mimeType": "application/vnd.google-apps.folder",
                "parents": [eval_id]
            },
            fields="id", supportsAllDrives=True
        ).execute()["id"]

        for key, fpath in files.items():
            fname = Path(fpath).name
            media = MediaFileUpload(fpath, resumable=True)
            service.files().create(
                body={"name": fname, "parents": [run_folder_id]},
                media_body=media, fields="id", supportsAllDrives=True
            ).execute()
        print(f"Reports uploaded to Drive: {EVAL_FOLDER_NAME}/{run_name}/")


## Run the evaluation
Timestamped run name, the main driver, and the actual call.


In [ ]:

# Timestamped run name

def evaluation_run_name():
    # timestamped run name encoding page scope (eval_all_ vs eval_pages_...)
    ts = datetime.now().strftime("%Y%m%d_%H%M")
    if PAGES == "all":
        return f"eval_all_{ts}"
    if isinstance(PAGES, list):
        pages = "_".join(map(str, PAGES[:10]))
        if len(PAGES) > 10:
            pages += "_etc"
        return f"eval_pages_{pages}_{ts}"
    if isinstance(PAGES, range):
        return f"eval_pages_{PAGES.start}_{PAGES.stop - 1}_{ts}"
    return f"eval_custom_{ts}"




In [ ]:
# Outputs: eval_NAME_results.csv (per-file metrics), eval_NAME_report.md (per-page detail), eval_NAME_page_summary.csv (per-page triage).
# Main evaluation driver

def run_evaluation():
    service  = get_drive_service() if SAVE_MODE in ("drive", "both") else None
    run_name = evaluation_run_name()

    if SAVE_MODE in ("drive", "both"):
        root_id    = find_folder(service, ROOT_FOLDER_NAME)
        results_id = find_folder(service, RESULTS_FOLDER, root_id)
    else:
        root_id    = None
        results_id = None

    results        = []
    markdown_lines = ["# OCR Evaluation Report", ""]

    log_data = load_run_logs(service, root_id, results_id=results_id)
    log_index, log_issues, page_run_map = build_log_index(log_data)

    if log_issues:
        markdown_lines.append("\n## Log Anomalies\n")
        for issue in log_issues:
            markdown_lines.append(f"- {issue}")
        markdown_lines.append("")

    # ── List page folders ──
    if SAVE_MODE in ("drive", "both"):
        page_folders = [
            f for f in list_files_in_folder(service, results_id)
            if f["mimeType"] == "application/vnd.google-apps.folder"
        ]
        print(f"Found {len(page_folders)} page folders in llm_ocr_results")
    else:
        page_folders = [
            {"name": p.name, "id": None, "path": p}
            for p in LOCAL_ROOT.iterdir()
            if p.is_dir() and re.match(r'^page_\d+$', p.name)
        ]

    total_pages          = 0
    propagation_failures = 0

    for page in sorted(
    page_folders,
    key=lambda x: int(re.search(r'(\d+)', x["name"]).group(1))
    if re.search(r'(\d+)', x["name"]) else 9999
):
        page_name      = page["name"]
        page_num_match = re.search(r'(\d+)', page_name)
        if not page_num_match:
            print(f"\n=== Skipping {page_name} (no page number found) ===")
            continue

        page_num = int(page_num_match.group())

        if PAGES != "all":
            if isinstance(PAGES, list) and page_num not in PAGES:
                continue
            if isinstance(PAGES, range) and page_num not in PAGES:
                continue

        if get_page_content_type(page_num) == "skip":
            print(f"\n=== Skipping {page_name} (cover/title page) ===")
            continue

        total_pages += 1
        print(f"\n=== Processing {page_name} (page {page_num}) ===")

        # ── List files in page folder ──
        if SAVE_MODE in ("drive", "both"):
            all_page_files = list_files_in_folder(service, page['id'])
        else:
            all_page_files = [
                {"name": f.name, "id": None, "createdTime": "",
                 "path": str(f)}
                for f in page["path"].iterdir()
                if f.is_file()
            ]

        # ── Build run maps ──
        ocr_runs = {}
        csv_runs = {}
        for f in all_page_files:
            fname = f["name"]
            m_ocr = re.search(r'_ocr_(\d+)\.txt$', fname)
            if m_ocr:
                run_n = int(m_ocr.group(1))
                if run_n not in ocr_runs or f.get("createdTime", "") > ocr_runs[run_n].get("createdTime", ""):
                    ocr_runs[run_n] = f
                continue
            m_csv = re.search(r'_semantic_(\d+)\.csv$', fname)
            if m_csv:
                run_n = int(m_csv.group(1))
                if run_n not in csv_runs or f.get("createdTime", "") > csv_runs[run_n].get("createdTime", ""):
                    csv_runs[run_n] = f

        # ── Flag unpaired files ──
        all_run_ids = set(ocr_runs.keys()) | set(csv_runs.keys())
        for run_n in sorted(all_run_ids):
            has_ocr = run_n in ocr_runs
            has_csv = run_n in csv_runs
            if has_ocr and not has_csv:
                results.append({
                    "Page": page_name, "Run": "MISSING", "Rows": 0,
                    "Issues": f"OCR_WITHOUT_CSV:run_{run_n}",
                    "Failure Reasons": f"OCR_WITHOUT_CSV:run_{run_n}"
                })
                markdown_lines.append(
                    f"- **{page_name}**: run {run_n} has OCR txt but no semantic CSV"
                )
            elif has_csv and not has_ocr:
                results.append({
                    "Page": page_name, "Run": "MISSING", "Rows": 0,
                    "Issues": f"CSV_WITHOUT_OCR:run_{run_n}",
                    "Failure Reasons": f"CSV_WITHOUT_OCR:run_{run_n}"
                })
                markdown_lines.append(
                    f"- **{page_name}**: run {run_n} has semantic CSV but no OCR txt"
                )

        semantic_files = [csv_runs[r] for r in sorted(csv_runs.keys())]
        if not semantic_files:
            results.append({
                "Page": page_name, "Run": "MISSING", "Rows": 0,
                "Issues": "NO_SEMANTIC_CSV",
                "Failure Reasons": "NO_SEMANTIC_CSV"
            })
            markdown_lines.append(f"- **{page_name}**: NO SEMANTIC CSV FOUND")
            continue

        page_metrics = []
        gt_text, gt_csv_text = load_ground_truth(page_name)

        for sem in semantic_files:
            file_name = sem["name"]

            if SAVE_MODE in ("drive", "both") and sem.get("id"):
                csv_text = download_text(service, sem["id"])
            else:
                csv_text = Path(sem["path"]).read_text(encoding="utf-8")

            metrics = analyze_csv(csv_text, page_num)

            try:
                prop_signal = (
                    check_provincia_signal(csv_text)
                    if page_num and 494 <= page_num <= 570
                    else check_propagation_signal(csv_text, page_num)
                )
            except Exception as e:
                print(f"  WARNING: propagation check failed for {file_name}: {e}")
                prop_signal = {
                    "propagation_flag": False, "empty_address_rows": 0,
                    "empty_address_pct": 0.0, "note": ""
                }

            metrics["propagation_signal"] = prop_signal
            page_metrics.append(metrics)

            run_id_match = re.search(r'_(\d+)\.csv', file_name)
            run_id   = int(run_id_match.group(1)) if run_id_match else None
            log_meta = get_best_log_meta(log_index, run_id, page_num)

            print(
                f"  {file_name} | rows={metrics.get('rows', 0)} | "
                f"layout={log_meta.get('layout', '?')} | "
                f"prop_flag={prop_signal.get('propagation_flag', False)}"
            )

            ocr_text = load_ocr_text(service, page.get("id"), page_name, file_name)
            if ocr_text:
                print(f"  OCR text loaded: {len(ocr_text)} chars")
            else:
                print(f"  WARNING: no OCR text for {file_name} — text metrics will be None")

            gt_metrics = evaluate_against_ground_truth(
                ocr_text=ocr_text, gt_text=gt_text,
                ocr_csv_text=csv_text, gt_csv_text=gt_csv_text
            )

            results.append({
                "Page":              page_name,
                "Run":               file_name,
                "Rows":              metrics.get("rows", 0),
                "Empty Rows":        metrics.get("empty_rows", 0),
                "Truncated Names":   metrics.get("truncated_names", 0),
                "Issues":            "; ".join(metrics.get("issues", [])),
                "Layout":            log_meta.get("layout", ""),
                "Ad Type":           log_meta.get("ad_type", ""),
                "Inferred Schema":   log_meta.get("schema", ""),
                "Flags":             ", ".join(log_meta.get("flags", [])),
                "Warning Count":     len(log_meta.get("warnings", [])),
                "Propagation Flag":  prop_signal.get("propagation_flag", False),
                "Propagation Note":  prop_signal.get("note", ""),
                **gt_metrics
            })

        # ── Markdown detail per page ──
        markdown_lines.append(f"\n### {page_name}\n")
        for idx, sem in enumerate(semantic_files):
            fname        = sem["name"]
            run_id_match = re.search(r'_(\d+)\.csv', fname)
            rid          = int(run_id_match.group(1)) if run_id_match else None
            lm           = get_best_log_meta(log_index, rid, page_num)
            m            = page_metrics[idx] if idx < len(page_metrics) else {}
            prop         = m.get("propagation_signal") or {}

            markdown_lines.append(f"**{fname}**")
            markdown_lines.append(
                f"- Rows: {m.get('rows', 0)} | "
                f"Empty: {m.get('empty_rows', 0)} | "
                f"Truncated: {m.get('truncated_names', 0)}"
            )
            markdown_lines.append(
                f"- Layout: {lm.get('layout', 'n/a')} | "
                f"Ad type: {lm.get('ad_type', 'n/a')}"
            )
            markdown_lines.append(f"- Schema: {lm.get('schema', 'n/a')}")
            markdown_lines.append(
                f"- Flags: {', '.join(lm.get('flags', [])) or 'none'}"
            )
            if m.get("issues"):
                markdown_lines.append(f"- Issues: {'; '.join(m['issues'])}")
            if prop.get("propagation_flag"):
                markdown_lines.append(
                    f"- ⚠ PROPAGATION (final CSV): {prop.get('note', '')}"
                )
            markdown_lines.append("")

        # ── Compare runs ──
        if len(page_metrics) > 1:
            for i in range(len(page_metrics) - 1):
                col_diff = (
                    page_metrics[i].get("columns", set()) ^
                    page_metrics[i + 1].get("columns", set())
                )
                rows_a   = page_metrics[i].get("rows", 0)
                rows_b   = page_metrics[i + 1].get("rows", 0)
                row_diff = abs(rows_a - rows_b)

                try:
                    csv_a  = semantic_files[i]
                    csv_b  = semantic_files[i + 1]
                    text_a = (
                        download_text(service, csv_a["id"])
                        if SAVE_MODE in ("drive", "both") and csv_a.get("id")
                        else Path(csv_a["path"]).read_text(encoding="utf-8")
                    )
                    text_b = (
                        download_text(service, csv_b["id"])
                        if SAVE_MODE in ("drive", "both") and csv_b.get("id")
                        else Path(csv_b["path"]).read_text(encoding="utf-8")
                    )
                    rows_parsed_a = list(csv.DictReader(io.StringIO(text_a)))
                    rows_parsed_b = list(csv.DictReader(io.StringIO(text_b)))
                    vals_a = set(
                        normalize(v)
                        for r in rows_parsed_a for v in r.values()
                        if v and str(v).strip()
                    )
                    vals_b = set(
                        normalize(v)
                        for r in rows_parsed_b for v in r.values()
                        if v and str(v).strip()
                    )
                    overlap = len(vals_a & vals_b)
                    union   = len(vals_a | vals_b)
                    value_similarity = round(100 * overlap / union, 1) if union else 100.0
                except Exception:
                    value_similarity = None

                results.append({
                    "Page":               page_name,
                    "Run":                f"compare_{i+1}_{i+2}",
                    "Column Differences": ", ".join(sorted(col_diff)) if col_diff else "none",
                    "Row Count Diff":     row_diff,
                    "Rows A":             rows_a,
                    "Rows B":             rows_b,
                    "Value Similarity %": value_similarity,
                })

        # ── Failure classification ──
        log_meta_list = []
        for sem in semantic_files:
            fname        = sem["name"]
            run_id_match = re.search(r'_(\d+)\.csv', fname)
            run_id       = int(run_id_match.group(1)) if run_id_match else None
            log_meta_list.append(get_best_log_meta(log_index, run_id, page_num))

        run_id_list = []
        for sem in semantic_files:
            fname        = sem["name"]
            run_id_match = re.search(r'_(\d+)\.csv', fname)
            run_id_list.append(
                int(run_id_match.group(1)) if run_id_match else None
            )
        page_failure_reasons = classify_page_failure(
            page_metrics, log_meta_list, run_ids=run_id_list
        )
        results.append({
            "Page": page_name,
            "Run":  "PAGE_SUMMARY",
            "Failure Reasons": ", ".join(page_failure_reasons)
        })

        if page_metrics and page_metrics[-1].get(
            "propagation_signal", {}
        ).get("propagation_flag", False):
            propagation_failures += 1

    # Assemble the final markdown report

    markdown_lines.append("\n## Summary\n")
    markdown_lines.append(f"- Pages evaluated: {total_pages}")

    prop_flagged_count = len(set(
        r["Page"] for r in results
        if r.get("Propagation Flag")
        and r.get("Run") not in ("PAGE_SUMMARY", "MISSING", "PROVINCIA_AGGREGATE")
        and not r.get("Run", "").startswith("compare_")
    ))
    missing_count = sum(1 for r in results if r.get("Run") == "MISSING")

    markdown_lines.append(f"- Pages with propagation flags: {prop_flagged_count}")
    markdown_lines.append(f"- Pages missing files: {missing_count}")
    markdown_lines.append("")

    INFORMATIONAL_ONLY = {"FULL_AD_PAGE"}
    real_failures = [
        r for r in results
        if r.get("Run") == "PAGE_SUMMARY"
        and r.get("Failure Reasons")
        and not all(
            reason.strip() in INFORMATIONAL_ONLY
            for reason in r["Failure Reasons"].split(",")
        )
    ]

    markdown_lines.append("\n## Pages Needing Attention\n")
    markdown_lines.append(
        "Only real structural errors listed here. "
        "Complex layout and full-ad pages are not errors.\n"
    )
    if real_failures:
        for r in real_failures:
            markdown_lines.append(f"- **{r['Page']}**: {r['Failure Reasons']}")
    else:
        markdown_lines.append("- None")

    markdown_lines.append("\n## Missing Files\n")
    missing_pages = [r["Page"] for r in results if r.get("Run") == "MISSING"]
    if missing_pages:
        for p in missing_pages:
            markdown_lines.append(f"- {p}")
    else:
        markdown_lines.append("- None")

    markdown_lines.append("\n## Address/Location Coverage\n")
    markdown_lines.append(
        "Counts from the final saved CSV after propagation ran. "
        "A flag means >10% of rows have neither Address nor Location.\n"
    )

    seen_pages    = set()
    flagged_lines = []
    clean_lines   = []

    for r in results:
        run = r.get("Run", "")
        if run in ("PAGE_SUMMARY", "MISSING") or run.startswith("compare_") or run == "PROVINCIA_AGGREGATE":
            continue
        page = r["Page"]
        if page in seen_pages:
            continue
        seen_pages.add(page)
        line = f"- **{page}**: {r.get('Rows', 0)} rows"
        if r.get("Propagation Flag"):
            flagged_lines.append(line + f" | {r.get('Propagation Note', '')} ⚠")
        else:
            clean_lines.append(line + f" | {r.get('Propagation Note', '')}")

    if flagged_lines:
        markdown_lines.append("### Flagged (review needed)\n")
        markdown_lines.extend(flagged_lines)
        markdown_lines.append("")
    if clean_lines:
        markdown_lines.append("### Clean\n")
        markdown_lines.extend(clean_lines)

    markdown_lines.append("\n## Ground Truth Results\n")
    gt_pages_found = set()
    for r in results:
        run = r.get("Run", "")
        if run in ("PAGE_SUMMARY", "MISSING") or run.startswith("compare_"):
            continue
        if r.get("semantic_f1") is not None or r.get("char_accuracy") is not None:
            gt_pages_found.add(r["Page"])

    markdown_lines.append(f"Pages with ground truth: {len(gt_pages_found)}\n")
    for page in sorted(gt_pages_found):
        page_runs = [
            r for r in results
            if r["Page"] == page
            and r.get("Run") not in ("PAGE_SUMMARY", "MISSING")
            and not r.get("Run", "").startswith("compare_")
            and r.get("semantic_f1") is not None
        ]
        if not page_runs:
            continue
        best = max(page_runs, key=lambda r: r["semantic_f1"])
        markdown_lines.append(
            f"- **{page}** (best run): "
            f"char_acc={best.get('char_accuracy', 'n/a')} | "
            f"bow={best.get('bow_similarity', 'n/a')} | "
            f"F1={best['semantic_f1']} | "
            f"col_aware_F1={best.get('semantic_f1_col_aware', 'n/a')} | "
            f"name_recall={best.get('name_recall', 'n/a')} | "
            f"row_similarity={best.get('row_count_similarity', 'n/a')}"
        )

    evaluate_provincia_aggregates(service, root_id, results, markdown_lines, PAGES)

    summary_df = build_summary_table(results, markdown_lines)
    markdown_lines.append("\n## Page Summary Table\n")
    markdown_lines.append(
        f"Full summary saved as `{run_name}_page_summary.csv` — "
        f"{len(summary_df)} pages, sorted by priority. "
        f"Filter `Action != OK` for pages needing review.\n"
    )
    if not summary_df.empty:
        for action, count in sorted(
            summary_df["Action"].value_counts().to_dict().items(),
            key=lambda x: x[1], reverse=True
        ):
            markdown_lines.append(f"- {action}: {count} pages")

    files = save_reports(
        results, "\n".join(markdown_lines), run_name, summary_df=summary_df
    )
    save_evaluation_reports(service, files, run_name)

    print(f"Done. Evaluated {total_pages} pages.")

In [2]:
run_evaluation()

Loading logs from Drive: logs/ subfolder under root
Found 571 page folders in llm_ocr_results

=== Processing page_41 (page 41) ===
  page_41_semantic_1.csv | rows=1 | layout=None | prop_flag=False
  OCR txt found: page_41_ocr_1.txt
  OCR text loaded: 240 chars
  page_41_semantic_2.csv | rows=2 | layout=COMPLEX_LAYOUT | prop_flag=False
  OCR txt found: page_41_ocr_2.txt
  OCR text loaded: 226 chars

=== Processing page_130 (page 130) ===
  page_130_semantic_1.csv | rows=2 | layout=COMPLEX_LAYOUT | prop_flag=False
  OCR txt found: page_130_ocr_1.txt
  OCR text loaded: 484 chars

=== Processing page_161 (page 161) ===
  page_161_semantic_1.csv | rows=0 | layout=None | prop_flag=False
  OCR txt found: page_161_ocr_1.txt
  OCR text loaded: 51 chars

=== Processing page_170 (page 170) ===
  page_170_semantic_1.csv | rows=1 | layout=None | prop_flag=False
  OCR txt found: page_170_ocr_1.txt
  OCR text loaded: 55 chars
  page_170_semantic_2.csv | rows=1 | layout=SIMPLE_LAYOUT | prop_flag=True

## Merge a partial re-run into the latest full eval


In [4]:
def merge_partial_eval_into_full(service, root_id, rerun_pages):
    # merges a partial re-run's eval into the latest full eval instead of re-evaluating all pages

    print("Loading latest full eval...")

    # ── Find latest full eval ──
    def get_subfolders(folder_id):
        return [
            f for f in list_files_in_folder(service, folder_id)
            if f["mimeType"] == "application/vnd.google-apps.folder"
        ]

    if SAVE_MODE in ("drive", "both") and service:
        eval_folder_id = find_folder(service, EVAL_FOLDER_NAME, root_id)
        subfolders     = get_subfolders(eval_folder_id)
        full_folders   = [f for f in subfolders if f["name"].startswith("eval_all_")]
        partial_folders = [f for f in subfolders
                           if any(p.replace("page_", "") in f["name"]
                                  for p in rerun_pages)]

        if not full_folders:
            print("ERROR: no eval_all_ folder found")
            return
        if not partial_folders:
            print("ERROR: no partial eval folder found for re-run pages")
            return

        latest_full    = max(full_folders,    key=lambda f: f["createdTime"])
        latest_partial = max(partial_folders, key=lambda f: f["createdTime"])
        print(f"  Full eval:    {latest_full['name']}")
        print(f"  Partial eval: {latest_partial['name']}")

        # Load full results CSV
        full_files    = list_files_in_folder(service, latest_full["id"])
        partial_files = list_files_in_folder(service, latest_partial["id"])

        def load_csv_from_files(files, suffix):
            f = next((x for x in files if x["name"].endswith(suffix)), None)
            if not f:
                return None
            return pd.read_csv(io.StringIO(download_text(service, f["id"])))

        full_results    = load_csv_from_files(full_files,    "_results.csv")
        full_summary    = load_csv_from_files(full_files,    "_page_summary.csv")
        partial_results = load_csv_from_files(partial_files, "_results.csv")
        partial_summary = load_csv_from_files(partial_files, "_page_summary.csv")

    else:
        # Local mode
        full_dirs    = sorted(
            [d for d in LOCAL_EVAL.iterdir()
             if d.is_dir() and d.name.startswith("eval_all_")],
            key=lambda d: d.stat().st_mtime
        )
        partial_dirs = sorted(
            [d for d in LOCAL_EVAL.iterdir()
             if d.is_dir() and not d.name.startswith("eval_all_")
             and d.name.startswith("eval_")],
            key=lambda d: d.stat().st_mtime
        )

        if not full_dirs or not partial_dirs:
            print("ERROR: could not find required eval folders locally")
            return

        latest_full    = full_dirs[-1]
        latest_partial = partial_dirs[-1]
        print(f"  Full eval:    {latest_full.name}")
        print(f"  Partial eval: {latest_partial.name}")

        full_results    = pd.read_csv(next(latest_full.glob("*_results.csv")))
        full_summary    = pd.read_csv(next(latest_full.glob("*_page_summary.csv")))
        partial_results = pd.read_csv(next(latest_partial.glob("*_results.csv")))
        partial_summary = pd.read_csv(next(latest_partial.glob("*_page_summary.csv")))

    if full_results is None or partial_results is None:
        print("ERROR: could not load results CSVs")
        return

    # ── Merge ──
    # Remove re-run pages from full eval
    merged_results = full_results[~full_results["Page"].isin(rerun_pages)].copy()
    merged_summary = full_summary[~full_summary["Page"].isin(rerun_pages)].copy()

    # Append new rows from partial eval
    merged_results = pd.concat([merged_results, partial_results], ignore_index=True)
    merged_summary = pd.concat([merged_summary, partial_summary], ignore_index=True)

    print(f"  Merged results: {len(merged_results)} rows "
          f"({len(partial_results)} new from partial)")

    # ── Save as new eval_all_ folder ──
    ts       = datetime.now().strftime("%Y%m%d_%H%M")
    run_name = f"eval_all_{ts}_merged"

    base_dir = Path(tempfile.mkdtemp())
    results_path = base_dir / f"{run_name}_results.csv"
    summary_path = base_dir / f"{run_name}_page_summary.csv"
    merged_results.to_csv(results_path, index=False)
    merged_summary.to_csv(summary_path, index=False)

    # Re-use save_evaluation_reports to save to correct location
    files = {
        "csv":     str(results_path),
        "md":      str(base_dir / f"{run_name}_report.md"),  # placeholder
        "summary": str(summary_path),
    }
    # Write placeholder md
    (base_dir / f"{run_name}_report.md").write_text(
        f"# Merged eval\nGenerated from {latest_full['name'] if SAVE_MODE in ('drive','both') else latest_full.name} "
        f"+ partial re-run of {rerun_pages}",
        encoding="utf-8"
    )

    save_evaluation_reports(service, files, run_name)
    print(f"Done. New merged eval saved as: {run_name}")
    print("Run version selector — it will auto-detect this as the latest full eval.")

In [ ]:

service  = get_drive_service() if SAVE_MODE in ("drive", "both") else None
root_id  = find_folder(service, ROOT_FOLDER_NAME) if service else None

# Page names re-run and being merged back in
rerun_pages = (
    [f"page_{n}" for n in range(494, 571)] +
    [f"page_{n}" for n in [41, 130, 161, 170, 320, 367, 383, 458]]
)

merge_partial_eval_into_full(
    service     = service,
    root_id     = root_id,
    rerun_pages = rerun_pages
)


Loading latest full eval...
  Full eval:    eval_all_20260612_1130
  Partial eval: eval_pages_494_495_496_497_498_499_500_501_502_503_etc_20260629_1417
  Merged results: 3476 rows (563 new from partial)
Reports uploaded to Drive: evaluation/eval_all_20260629_1447_merged/
Done. New merged eval saved as: eval_all_20260629_1447_merged
Run version selector — it will auto-detect this as the latest full eval.
